# FinMiniLM Fine-Tune — Colab Version (GPU)

## Before running:
1. **Runtime → Change runtime type → T4 GPU**
2. Upload `training_pairs.csv` and `s4_corpus.csv` to `/content/` (Files panel on left)
3. Run all cells in order

## What this does:
- Stratified 80/20 train/eval split
- Hard negative mining (FAISS top-30)
- Fine-tune MiniLM with MultipleNegativesRankingLoss
- Evaluate NDCG@10 / Recall@10 / MRR after each epoch
- Save best checkpoint

## Expected time on T4 GPU: ~15-20 min

In [1]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name     :', torch.cuda.get_device_name(0))
    print('GPU memory   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected — go to Runtime > Change runtime type > T4 GPU')

GPU available: True
GPU name     : Tesla T4
GPU memory   : 15.6 GB


In [2]:
# ── Cell 2: Install ───────────────────────────────────────────────────────────
!pip install -q sentence-transformers faiss-cpu scikit-learn datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.1 MB/s eta 0:00:00


In [3]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import os, json, random
import numpy as np
import pandas as pd
import faiss
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ── Paths (Colab) ─────────────────────────────────────────────────────────────
PAIRS_PATH     = '/content/training_pairs.csv'
S4_CORPUS_PATH = '/content/s4_corpus.csv'
MODEL_SAVE     = '/content/minilm_finetuned'
RESULTS_DIR    = '/content/Results'
MINILM_NAME    = 'sentence-transformers/all-MiniLM-L6-v2'

os.makedirs(MODEL_SAVE,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(PAIRS_PATH),     'Upload training_pairs.csv to /content/'
assert os.path.exists(S4_CORPUS_PATH), 'Upload s4_corpus.csv to /content/'

print('Imports OK')
print('Device :', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Pairs  :', PAIRS_PATH)
print('Corpus :', S4_CORPUS_PATH)

Imports OK
Device : cuda
Pairs  : /content/training_pairs.csv
Corpus : /content/s4_corpus.csv


In [5]:
# ── Cell 4: Load Data + Stratified 80/20 Split ────────────────────────────────
pairs_df  = pd.read_csv(PAIRS_PATH)
corpus_df = pd.read_csv(S4_CORPUS_PATH)

print(f'Total pairs : {len(pairs_df):,}')
print(f'Corpus      : {len(corpus_df):,} chunks')
print('\nFull category distribution:')
print(pairs_df['category'].value_counts().to_string())

# Stratified split: 80% train, 20% eval
train_df, eval_df = train_test_split(
    pairs_df,
    test_size=0.20,
    random_state=42,
    stratify=pairs_df['category']
)
train_df = train_df.reset_index(drop=True)
eval_df  = eval_df.reset_index(drop=True)

print(f'\nTrain : {len(train_df):,} pairs (80%)')
print(train_df['category'].value_counts().to_string())
print(f'\nEval  : {len(eval_df):,} pairs (20%) — held out')
print(eval_df['category'].value_counts().to_string())

Total pairs : 17,682
Corpus      : 14,502 chunks

Full category distribution:
category
Regulatory             7941
Payment_Industry       5244
Consumer_Protection    2502
Synthetic_Policies     1995

Train : 14,145 pairs (80%)
category
Regulatory             6353
Payment_Industry       4195
Consumer_Protection    2001
Synthetic_Policies     1596

Eval  : 3,537 pairs (20%) — held out
category
Regulatory             1588
Payment_Industry       1049
Consumer_Protection     501
Synthetic_Policies      399


In [6]:
# ── Cell 5: Hard Negative Mining ──────────────────────────────────────────────
TOP_K_NEGATIVES = 30

print('Loading base MiniLM...')
base_model   = SentenceTransformer(MINILM_NAME)
corpus_texts = corpus_df['text'].tolist()
corpus_ids   = corpus_df['chunk_id'].tolist()

print(f'Encoding {len(corpus_texts):,} corpus chunks...')
corpus_embs = base_model.encode(
    corpus_texts,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)

index = faiss.IndexFlatIP(corpus_embs.shape[1])
index.add(corpus_embs)
print(f'FAISS index: {index.ntotal:,} vectors')

print(f'Encoding {len(train_df):,} train questions...')
q_embs = base_model.encode(
    train_df['question'].tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)

_, indices = index.search(q_embs, TOP_K_NEGATIVES)

triplets   = []
pairs_only = []

for i, row in enumerate(train_df.itertuples()):
    hard_neg_candidates = []
    for corpus_idx in indices[i]:
        cid = corpus_ids[corpus_idx]
        if cid != row.chunk_id:
            hard_neg_candidates.append(corpus_texts[corpus_idx])
        if len(hard_neg_candidates) >= 5:
            break
    if hard_neg_candidates:
        triplets.append((row.question, row.chunk_text, random.choice(hard_neg_candidates)))
    else:
        pairs_only.append((row.question, row.chunk_text))

coverage = len(triplets) / max(len(train_df), 1) * 100
print(f'\nTriplets (with hard neg) : {len(triplets):,}  ({coverage:.1f}%)')
print(f'Pairs    (no hard neg)   : {len(pairs_only):,}')

Loading base MiniLM...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Encoding 14,502 corpus chunks...


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

FAISS index: 14,502 vectors
Encoding 14,145 train questions...


Batches:   0%|          | 0/56 [00:00<?, ?it/s]


Triplets (with hard neg) : 14,145  (100.0%)
Pairs    (no hard neg)   : 0


In [7]:
# ── Cell 6: Build InformationRetrievalEvaluator ───────────────────────────────
eval_queries  = {}
eval_relevant = {}

for i, row in enumerate(eval_df.itertuples()):
    qid = f'eval_q_{i:05d}'
    eval_queries[qid]  = row.question
    eval_relevant[qid] = {row.chunk_id}

eval_corpus = dict(zip(corpus_df['chunk_id'], corpus_df['text']))

print(f'Evaluator:')
print(f'  Queries      : {len(eval_queries):,}')
print(f'  Corpus pool  : {len(eval_corpus):,} chunks')

evaluator = InformationRetrievalEvaluator(
    queries=eval_queries,
    corpus=eval_corpus,
    relevant_docs=eval_relevant,
    name='fineval',
    batch_size=256,
    show_progress_bar=True,
)
print('Evaluator ready.')

Evaluator:
  Queries      : 3,537
  Corpus pool  : 14,502 chunks
Evaluator ready.


In [8]:
# ── Cell 7: Fine-Tune ─────────────────────────────────────────────────────────
EPOCHS     = 3
BATCH_SIZE = 64    # larger on GPU
LR         = 2e-5

train_examples = []
for q, pos, neg in triplets:
    train_examples.append(InputExample(texts=[q, pos, neg]))
for q, pos in pairs_only:
    train_examples.append(InputExample(texts=[q, pos]))
random.shuffle(train_examples)

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
    drop_last=True,
    num_workers=2,
)

WARMUP = int(len(train_dataloader) * 0.1)

print('Training config:')
print(f'  Total examples : {len(train_examples):,}')
print(f'  Steps/epoch    : {len(train_dataloader):,}')
print(f'  Epochs         : {EPOCHS}')
print(f'  Batch size     : {BATCH_SIZE}')
print(f'  Learning rate  : {LR}')
print(f'  Warmup steps   : {WARMUP}')

model      = SentenceTransformer(MINILM_NAME)
train_loss = losses.MultipleNegativesRankingLoss(model)

print('\nStarting fine-tuning...')
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    evaluator=evaluator,
    evaluation_steps=0,       # evaluate at end of each epoch only
    output_path=MODEL_SAVE,
    save_best_model=True,
    show_progress_bar=True,
    optimizer_params={'lr': LR},
)

print(f'\nFine-tuning complete!')
print(f'Model saved to: {MODEL_SAVE}')

Training config:
  Total examples : 14,145
  Steps/epoch    : 221
  Epochs         : 3
  Batch size     : 64
  Learning rate  : 2e-05
  Warmup steps   : 22


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Starting fine-tuning...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

...

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:32<00:00, 32.52s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning complete!
Model saved to: /content/minilm_finetuned


In [9]:
# ── Cell 8: Show Eval Results ─────────────────────────────────────────────────
import glob

eval_files = glob.glob(os.path.join(MODEL_SAVE, 'eval', '*.csv'))
if eval_files:
    results_df = pd.read_csv(eval_files[0])
    print('Evaluation results per epoch:')
    print(results_df.to_string(index=False))
else:
    print('No eval results file found — check', MODEL_SAVE, '/eval/')

Evaluation results per epoch:
 epoch  steps  cosine-Accuracy@1  cosine-Accuracy@3  cosine-Accuracy@5  cosine-Accuracy@10  cosine-Precision@1  cosine-Recall@1  cosine-Precision@3  cosine-Recall@3  cosine-Precision@5  cosine-Recall@5  cosine-Precision@10  cosine-Recall@10  cosine-MRR@10  cosine-NDCG@10  cosine-MAP@100
   1.0    221           0.564603           0.743568           0.804919            0.865988            0.564603         0.564603            0.247856         0.743568            0.160984         0.804919             0.086599          0.865988       0.667750        0.715867        0.672751
   2.0    442           0.573933           0.756008           0.813118            0.869946            0.573933         0.573933            0.252003         0.756008            0.162624         0.813118             0.086995          0.869946       0.676584        0.723677        0.682013
   3.0    663           0.576760           0.757987           0.813401            0.872491            0.57

In [10]:
# ── Cell 9: Download Fine-Tuned Model ─────────────────────────────────────────
# Zips the model folder and downloads it to your machine
# Then put the unzipped folder into: fine_tuning/minilm_finetuned/

import shutil
zip_path = '/content/minilm_finetuned'
shutil.make_archive(zip_path, 'zip', MODEL_SAVE)
print(f'Zipped to: {zip_path}.zip')

from google.colab import files
files.download(f'{zip_path}.zip')
print('Download started — save it and unzip into fine_tuning/minilm_finetuned/')

Zipped to: /content/minilm_finetuned.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started — save it and unzip into fine_tuning/minilm_finetuned/


In [ ]:
# ── Cell 10: Save Summary JSON ────────────────────────────────────────────────
summary = {
    'base_model'             : MINILM_NAME,
    'saved_to'               : MODEL_SAVE,
    'total_pairs'            : len(pairs_df),
    'train_pairs'            : len(train_df),
    'eval_pairs'             : len(eval_df),
    'triplets_with_hard_neg' : len(triplets),
    'pairs_without_hard_neg' : len(pairs_only),
    'hard_neg_coverage_pct'  : round(coverage, 2),
    'epochs'                 : EPOCHS,
    'batch_size'             : BATCH_SIZE,
    'learning_rate'          : LR,
    'loss'                   : 'MultipleNegativesRankingLoss',
    'evaluator'              : 'InformationRetrievalEvaluator (NDCG@10 per epoch)',
    'data_source'            : '17,682 chunk->question pairs from 41 financial PDFs',
}
with open(os.path.join(RESULTS_DIR, 'finetune_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary saved.')
for k, v in summary.items():
    print(f'  {k:<30} {v}')